In [1]:
import math
import random
from collections import Counter


class DecisionNode:
    """Class to represent a single node in a decision tree."""
    
    def __init__(self, left=None, right=None, feature=None, threshold=None, value=None):
        self.left = left
        self.right = right
        self.feature = feature
        self.threshold = threshold
        self.value = value

    def is_leaf(self):
        return self.value is not None

    def decide(self, features):
        if self.is_leaf():
            return self.value
        if features[self.feature] <= self.threshold:
            return self.left.decide(features)
        else:
            return self.right.decide(features)


# ==================== IMPURITY FUNCTIONS ====================

def gini_impurity(labels):
    n = len(labels)
    if n == 0:
        return 0.0
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return 1.0 - sum((c / n) ** 2 for c in counts.values())


def entropy(labels):
    n = len(labels)
    if n == 0:
        return 0.0
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return -sum((c / n) * math.log2(c / n) for c in counts.values() if c > 0)


def information_gain(parent_labels, left_labels, right_labels, criterion="gini"):
    measure = gini_impurity if criterion == "gini" else entropy
    n = len(parent_labels)
    n_left = len(left_labels)
    n_right = len(right_labels)
    if n_left == 0 or n_right == 0:
        return 0.0
    parent_impurity = measure(parent_labels)
    child_impurity = (n_left / n) * measure(left_labels) + (n_right / n) * measure(right_labels)
    return parent_impurity - child_impurity


def variance_reduction(parent_values, left_values, right_values):
    if len(left_values) == 0 or len(right_values) == 0:
        return 0.0
    n = len(parent_values)
    parent_var = _variance(parent_values)
    child_var = (len(left_values) / n) * _variance(left_values) + (len(right_values) / n) * _variance(right_values)
    return parent_var - child_var


def _variance(values):
    n = len(values)
    if n == 0:
        return 0.0
    mean = sum(values) / n
    return sum((v - mean) ** 2 for v in values) / n


def _mean(values):
    if len(values) == 0:
        return 0.0
    return sum(values) / len(values)


def majority_vote(labels):
    counts = Counter(labels)
    return counts.most_common(1)[0][0]


# ==================== DECISION TREE CLASS (embedded for forest) ====================

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, criterion="gini",
                 max_features=None, task="classification"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features
        self.task = task
        self.root = None
        self.feature_importances_ = None
        self.n_features = 0
        self.n_samples = 0

    def fit(self, X, y):
        self.n_features = len(X[0])
        self.feature_importances_ = [0.0] * self.n_features
        self.n_samples = len(X)
        self.root = self._build_tree(X, y, depth=0)
        total = sum(self.feature_importances_)
        if total > 0:
            self.feature_importances_ = [fi / total for fi in self.feature_importances_]

    def predict(self, X):
        return [self.root.decide(x) for x in X]

    def _build_tree(self, X, y, depth):
        if self.task == "classification":
            all_same = len(set(y)) == 1
        else:
            all_same = len(set(y)) == 1

        if all_same:
            return DecisionNode(value=y[0] if self.task == "classification" else _mean(y))

        if self.max_depth is not None and depth >= self.max_depth:
            return DecisionNode(value=majority_vote(y) if self.task == "classification" else _mean(y))

        if len(y) < self.min_samples_split:
            return DecisionNode(value=majority_vote(y) if self.task == "classification" else _mean(y))

        best_feature, best_threshold, best_gain = self._best_split(X, y)

        if best_feature is None or best_gain <= 0:
            return DecisionNode(value=majority_vote(y) if self.task == "classification" else _mean(y))

        left_X, left_y, right_X, right_y = self._split_data(X, y, best_feature, best_threshold)

        if len(left_y) < self.min_samples_leaf or len(right_y) < self.min_samples_leaf:
            return DecisionNode(value=majority_vote(y) if self.task == "classification" else _mean(y))

        weight = len(y) / self.n_samples
        self.feature_importances_[best_feature] += weight * best_gain

        depth += 1
        left_child = self._build_tree(left_X, left_y, depth)
        right_child = self._build_tree(right_X, right_y, depth)

        return DecisionNode(left=left_child, right=right_child,
                            feature=best_feature, threshold=best_threshold)

    def _best_split(self, X, y):
        best_feature = None
        best_threshold = None
        best_gain = -1.0

        if self.max_features is None:
            feature_indices = list(range(self.n_features))
        elif self.max_features == "sqrt":
            k = max(1, int(math.sqrt(self.n_features)))
            feature_indices = random.sample(range(self.n_features), k)
        elif isinstance(self.max_features, int):
            k = min(self.max_features, self.n_features)
            feature_indices = random.sample(range(self.n_features), k)
        else:
            feature_indices = list(range(self.n_features))

        for feature_idx in feature_indices:
            values = sorted(set(X[i][feature_idx] for i in range(len(X))))
            if len(values) <= 1:
                continue

            for i in range(len(values) - 1):
                threshold = (values[i] + values[i + 1]) / 2.0
                left_y = [y[j] for j in range(len(X)) if X[j][feature_idx] <= threshold]
                right_y = [y[j] for j in range(len(X)) if X[j][feature_idx] > threshold]

                if len(left_y) < self.min_samples_leaf or len(right_y) < self.min_samples_leaf:
                    continue

                if self.task == "classification":
                    gain = information_gain(y, left_y, right_y, self.criterion)
                else:
                    gain = variance_reduction(y, left_y, right_y)

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold, best_gain

    def _split_data(self, X, y, feature, threshold):
        left_X, left_y, right_X, right_y = [], [], [], []
        for i in range(len(X)):
            if X[i][feature] <= threshold:
                left_X.append(X[i])
                left_y.append(y[i])
            else:
                right_X.append(X[i])
                right_y.append(y[i])
        return left_X, left_y, right_X, right_y


# ==================== RANDOM FOREST CLASS ====================

class RandomForest:
    def __init__(self, n_trees=100, max_depth=None,
                 min_samples_split=2, max_features="sqrt",
                 criterion="gini", task="classification"):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.task = task
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        n = len(X)
        for _ in range(self.n_trees):
            indices = [random.randint(0, n - 1) for _ in range(n)]
            X_boot = [X[i] for i in indices]
            y_boot = [y[i] for i in indices]

            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                criterion=self.criterion,
                task=self.task,
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

    def predict(self, X):
        all_preds = [tree.predict(X) for tree in self.trees]
        predictions = []
        for i in range(len(X)):
            if self.task == "classification":
                votes = {}
                for preds in all_preds:
                    v = preds[i]
                    votes[v] = votes.get(v, 0) + 1
                predictions.append(max(votes, key=votes.get))
            else:
                predictions.append(sum(preds[i] for preds in all_preds) / len(all_preds))
        return predictions

    def feature_importances(self):
        n_features = self.trees[0].n_features
        importances = [0.0] * n_features
        for tree in self.trees:
            for j in range(n_features):
                importances[j] += tree.feature_importances_[j]
        total = sum(importances)
        if total > 0:
            importances = [imp / total for imp in importances]
        return importances


# ==================== DEMO (only if run directly) ====================

def accuracy(y_true, y_pred):
    return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

def generate_classification_data(n_samples=200, seed=42):
    random.seed(seed)
    X, y = [], []
    for _ in range(n_samples):
        x1 = random.uniform(-3, 3)
        x2 = random.uniform(-3, 3)
        noise = random.gauss(0, 0.3)
        if x1 ** 2 + x2 ** 2 + noise < 3:
            label = 0
        elif x1 + x2 + noise > 1:
            label = 1
        else:
            label = 2
        X.append([x1, x2])
        y.append(label)
    return X, y

def train_test_split(X, y, test_ratio=0.2, seed=42):
    random.seed(seed)
    n = len(X)
    indices = list(range(n))
    random.shuffle(indices)
    split = int(n * (1 - test_ratio))
    train_idx = indices[:split]
    test_idx = indices[split:]
    return [X[i] for i in train_idx], [y[i] for i in train_idx], [X[i] for i in test_idx], [y[i] for i in test_idx]

if __name__ == "__main__":
    
    print("RANDOM FOREST (Standalone)")

    X, y = generate_classification_data(300, seed=42)
    X_train, y_train, X_test, y_test = train_test_split(X, y)

    print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}\n")

    print("Comparing Single Tree vs Random Forest (50 trees, depth=5):")
    
    single_tree = DecisionTree(max_depth=5)
    single_tree.fit(X_train, y_train)
    single_acc = accuracy(y_test, single_tree.predict(X_test))
    
    rf = RandomForest(n_trees=50, max_depth=5)
    rf.fit(X_train, y_train)
    rf_acc = accuracy(y_test, rf.predict(X_test))
    
    print(f"  Single Tree Test Accuracy: {single_acc:.4f}")
    print(f"  Random Forest Test Accuracy: {rf_acc:.4f}")
    print(f"  Improvement: {(rf_acc - single_acc):.4f}\n")

    print("Effect of number of trees on Test Accuracy:")
    print(f"  {'N Trees':>8s}  {'Test Acc':>10s}")
    print(f"  {'-' * 8}  {'-' * 10}")
    for n in [1, 3, 5, 10, 25, 50, 100]:
        rf = RandomForest(n_trees=n, max_depth=5)
        rf.fit(X_train, y_train)
        acc = accuracy(y_test, rf.predict(X_test))
        print(f"  {n:>8d}  {acc:>10.4f}")
    
    print("\nFeature Importances (from 50 trees):")
    importances = rf.feature_importances()
    for i, imp in enumerate(importances):
        print(f"  Feature {i}: {imp:.4f}")

RANDOM FOREST (Standalone)
Training samples: 240, Test samples: 60

Comparing Single Tree vs Random Forest (50 trees, depth=5):
  Single Tree Test Accuracy: 0.9000
  Random Forest Test Accuracy: 0.9667
  Improvement: 0.0667

Effect of number of trees on Test Accuracy:
   N Trees    Test Acc
  --------  ----------
         1      0.7333
         3      0.8000
         5      0.9000
        10      0.9000
        25      0.9000
        50      0.9500
       100      0.9500

Feature Importances (from 50 trees):
  Feature 0: 0.5202
  Feature 1: 0.4798
